In [1]:
import paramiko
import os
import threading
import time
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(".env.vault")

def send_material():
    """
    Send the material file over SFTP.

    Parameters:
    - remote_material_path: The remote path where the file should be uploaded.
    - remote_host: The hostname of the remote server.
    - port: The port to connect to.
    - user: The username for authentication.
    - password: The password for authentication.
    - identityfile: The path to the private key file for authentication.
    """
    # Material path
    local_material_path = "/home/joffreyma/Projets/verbose-octo-carnival/data/dune/text/plot/dune_plot_summary.txt"

    # Remote paths
    remote_material_path = os.getenv('REMOTE_MATERIAL_PATH')

    # Remote connection variables
    gateway = os.getenv('GATEWAY')
    user = os.getenv('USERSSH')
    password = os.getenv('PASSWORD')
    identityfile = os.getenv('IDENTITYFILE')
    host = os.getenv('HOST')
    port = os.getenv('PORT')

    # Establish SSH connection
    pkey = paramiko.RSAKey.from_private_key_file(identityfile, password=password)
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    gw_client = paramiko.SSHClient()
    gw_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    gw_client.connect(hostname=gateway, port=port, username=user, password=password, sock=None, pkey=pkey)
    sock = gw_client.get_transport().open_channel(
            'direct-tcpip', (host, 22), ('', 0)
        )
    client.connect(hostname=host, port=port, username=user, password=password, sock=sock, pkey=pkey)

    # Open SFTP session
    sftp = client.open_sftp()

    # Transfer the material
    # Change the following line to send the file instead of getting it
    sftp.put(local_material_path, remote_material_path)

    # Close connections
    sftp.close()
    client.close()

In [2]:
send_material()